# Log Pengujian — Sistem Skrining Risiko Diabetes
**Puskesmas Harapan Sehat** · Pengujian artefak model (`.joblib`, `.json`) dan dataset Pima Indians Diabetes.

Notebook ini menjalankan **10 skenario pengujian** (TC-01 s.d. TC-10) dan menyusun log berisi: ID, skenario, input, hasil diharapkan, hasil aktual, status PASS/FAIL, dan waktu.

**Cara pakai:** letakkan notebook ini satu folder dengan `pima-indians-diabetes.csv`, `model_skrining_diabetes.joblib`, dan `model_web_bundle.json`, lalu jalankan semua sel (*Run All*). Di Google Colab, upload ketiga file tersebut ke sesi terlebih dahulu.

## 1. Konfigurasi & Pemuatan Artefak

In [1]:
import json, math, warnings, datetime
import numpy as np, pandas as pd, joblib, sklearn
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

JOBLIB = "model_skrining_diabetes.joblib"
BUNDLE = "model_web_bundle.json"
CSV    = "pima-indians-diabetes.csv"

COLS = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigreeFunction","Age","Outcome"]
ZERO_INVALID = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]

art    = joblib.load(JOBLIB)
bundle = json.load(open(BUNDLE))
df = pd.read_csv(CSV, header=None, names=COLS)
df[ZERO_INVALID] = df[ZERO_INVALID].replace(0, np.nan)   # 0 tidak masuk akal -> NaN (sesuai notebook pelatihan)

# Cek versi scikit-learn: artefak harus dimuat dengan versi yang sama saat pelatihan
VER_MISMATCH = (art["sklearn_version"], sklearn.__version__) if art["sklearn_version"] != sklearn.__version__ else None
print("scikit-learn artefak :", art["sklearn_version"])
print("scikit-learn terpasang:", sklearn.__version__)
if VER_MISMATCH:
    print("PERINGATAN: versi berbeda. Solusi benar: pip install scikit-learn==" + art["sklearn_version"])
    # Shim sementara agar pengujian tetap bisa berjalan pada versi sklearn yang lebih baru
    for _v in art["models"].values():
        for _, step in _v["pipeline"].steps:
            if hasattr(step, "_fit_dtype") and not hasattr(step, "_fill_dtype"):
                step._fill_dtype = step._fit_dtype

scikit-learn artefak : 1.6.1
scikit-learn terpasang: 1.9.1
PERINGATAN: versi berbeda. Solusi benar: pip install scikit-learn==1.6.1


## 2. Fungsi Bantu & Wadah Log

In [2]:
def predict(ver, patient):
    """Prediksi memakai pipeline joblib. Nilai 0 pada kolom lab/antropometri dianggap data hilang."""
    m = art["models"][ver]
    row = {}
    for f in m["features"]:
        v = patient.get(f, np.nan)
        try:
            v = float(v)
        except (TypeError, ValueError):
            raise ValueError(f"{f} bukan angka: {v!r}")
        if f in ZERO_INVALID and v == 0:
            v = np.nan
        row[f] = v
    p = float(m["pipeline"].predict_proba(pd.DataFrame([row])[m["features"]])[0, 1])
    return p, int(p >= m["default_threshold"])

def manual_logreg(ver, patient):
    """Hitung ulang regresi logistik langsung dari parameter JSON (tanpa sklearn)."""
    P = bundle["models"][ver]["parameters"]
    z = P["intercept"]
    for f in bundle["models"][ver]["features"]:
        v = patient.get(f, np.nan)
        if (f in ZERO_INVALID and v == 0) or v is None or (isinstance(v, float) and math.isnan(v)):
            v = P["medians"][f]
        z += P["coefficients"][f] * (v - P["scaler_mean"][f]) / P["scaler_scale"][f]
    return 1 / (1 + math.exp(-z))

log = []
def record(id_, nama, input_, expected, actual, ok):
    log.append({"ID": id_, "Skenario": nama, "Input": input_, "Hasil Diharapkan": expected,
                "Hasil Aktual": actual, "Status": "PASS" if ok else "FAIL",
                "Waktu": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")})
    print(f"{id_} [{'PASS' if ok else 'FAIL'}] {actual}")

## 3. Skenario Pengujian
### TC-01 — Integritas artefak model (joblib & JSON)
Memastikan struktur fitur, ambang keputusan, disclaimer, dan kesesuaian antara dua file artefak.

In [3]:
checks = {
    "versi_a 8 fitur": len(art["models"]["versi_a"]["features"]) == 8,
    "versi_b 6 fitur": len(art["models"]["versi_b"]["features"]) == 6,
    "versi_b tanpa Glucose/Insulin": not {"Glucose","Insulin"} & set(art["models"]["versi_b"]["features"]),
    "threshold A=0.35": art["models"]["versi_a"]["default_threshold"] == 0.35,
    "threshold B=0.40": art["models"]["versi_b"]["default_threshold"] == 0.40,
    "disclaimer ada": "BUKAN diagnosis" in art["disclaimer"],
    "joblib == json (threshold & fitur)": all(
        art["models"][v]["features"] == bundle["models"][v]["features"]
        and art["models"][v]["default_threshold"] == bundle["models"][v]["default_threshold"]
        for v in ("versi_a", "versi_b")),
}
record("TC-01", "Integritas artefak model (joblib & JSON)", "Muat 2 file artefak",
       "Semua 7 pemeriksaan struktur lolos",
       (f"PERINGATAN versi sklearn: artefak {VER_MISMATCH[0]} vs terpasang {VER_MISMATCH[1]} (model gagal prediksi tanpa shim). " if VER_MISMATCH else "")
       + f"{sum(checks.values())}/7 lolos"
       + ("" if all(checks.values()) else " | gagal: " + ", ".join(k for k, v in checks.items() if not v)),
       all(checks.values()))

TC-01 [PASS] PERINGATAN versi sklearn: artefak 1.6.1 vs terpasang 1.9.1 (model gagal prediksi tanpa shim). 7/7 lolos


### TC-02 — Validasi dataset & penanganan nilai 0

In [4]:
n_neg, n_pos = int((df.Outcome == 0).sum()), int((df.Outcome == 1).sum())
zero_counts = {c: int(df[c].isna().sum()) for c in ZERO_INVALID}
record("TC-02", "Validasi dataset & penanganan nilai 0", "pima-indians-diabetes.csv (768 baris)",
       "768 baris; 500 neg/268 pos; nilai 0 -> NaN pada 5 kolom",
       f"{len(df)} baris; {n_neg} neg/{n_pos} pos; NaN: {zero_counts}",
       len(df) == 768 and (n_neg, n_pos) == (500, 268))

TC-02 [PASS] 768 baris; 500 neg/268 pos; NaN: {'Glucose': 5, 'BloodPressure': 35, 'SkinThickness': 227, 'Insulin': 374, 'BMI': 11}


### TC-03 — Konsistensi prediksi joblib vs rumus logistik dari JSON

In [5]:
pat = {"Pregnancies":4, "Glucose":148, "BloodPressure":76, "SkinThickness":30,
       "Insulin":0, "BMI":33.6, "DiabetesPedigreeFunction":0.627, "Age":45}   # pasien contoh di notebook pelatihan
diffs = {}
for v in ("versi_a", "versi_b"):
    p_skl, _ = predict(v, pat)
    p_man = manual_logreg(v, pat)
    diffs[v] = (round(p_skl, 4), round(p_man, 4))
record("TC-03", "Konsistensi prediksi joblib vs rumus logistik dari JSON (pasien contoh notebook)", str(pat),
       "Selisih probabilitas < 0.0001 untuk Versi A & B",
       f"A: sklearn/manual={diffs['versi_a']}; B: {diffs['versi_b']}",
       all(abs(a - b) < 1e-4 for a, b in diffs.values()))

TC-03 [PASS] A: sklearn/manual=(0.7757, 0.7757); B: (0.6381, 0.6381)


### TC-04 — Input 0 pada kolom lab diperlakukan sebagai data hilang

In [6]:
a0, _ = predict("versi_a", pat)
an, _ = predict("versi_a", {**pat, "Insulin": np.nan})
record("TC-04", "Input 0 pada kolom lab diperlakukan sebagai data hilang",
       "Insulin=0 vs Insulin=NaN (fitur lain sama)", "Probabilitas identik",
       f"p(0)={a0:.4f}; p(NaN)={an:.4f}", abs(a0 - an) < 1e-12)

TC-04 [PASS] p(0)=0.7757; p(NaN)=0.7757


### TC-05 — Kasus tipikal berisiko tinggi vs rendah (Versi A)

In [7]:
hi = {"Pregnancies":8, "Glucose":183, "BloodPressure":64, "SkinThickness":0, "Insulin":0, "BMI":23.3, "DiabetesPedigreeFunction":0.672, "Age":32}  # baris 3 CSV, Outcome=1
lo = {"Pregnancies":1, "Glucose":85,  "BloodPressure":66, "SkinThickness":29, "Insulin":0, "BMI":26.6, "DiabetesPedigreeFunction":0.351, "Age":31}  # baris 2 CSV, Outcome=0
ph, dh = predict("versi_a", hi)
pl, dl = predict("versi_a", lo)
record("TC-05", "Kasus tipikal berisiko tinggi vs rendah (Versi A)",
       "Baris 3 CSV (Glukosa 183) vs baris 2 CSV (Glukosa 85)",
       "Tinggi -> rujuk (1); Rendah -> tidak rujuk (0)",
       f"Tinggi p={ph:.3f} -> {dh}; Rendah p={pl:.3f} -> {dl}", dh == 1 and dl == 0)

TC-05 [PASS] Tinggi p=0.864 -> 1; Rendah p=0.059 -> 0


### TC-06 — Konsistensi ambang keputusan & monotonisitas terhadap Glukosa

In [8]:
thr = art["models"]["versi_a"]["default_threshold"]
cases = {g: predict("versi_a", {**pat, "Glucose": g}) for g in np.arange(100, 141, 1.0)}
consistent = all((p >= thr) == bool(d) for p, d in cases.values())
keys = list(cases)
mono = all(cases[g][0] <= cases[g2][0] for g, g2 in zip(keys[:-1], keys[1:]))
record("TC-06", "Konsistensi ambang keputusan & monotonisitas terhadap Glukosa",
       "Glukosa 100-140 (langkah 1), fitur lain tetap",
       "Label = (p >= 0.35); p naik seiring glukosa",
       f"label konsisten={consistent}; monoton naik={mono}", consistent and mono)

TC-06 [PASS] label konsisten=True; monoton naik=True


### TC-07 — Reproduksi confusion matrix data uji (split 80:20, seed 42)

In [9]:
X = df[COLS[:-1]]
y = df.Outcome.astype(int)
_, Xte, _, yte = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

msgs, ok = [], True
for v in ("versi_a", "versi_b"):
    mm = art["models"][v]
    pr = mm["pipeline"].predict_proba(Xte[mm["features"]])[:, 1]
    tn, fp, fn, tp = confusion_matrix(yte, (pr >= mm["default_threshold"]).astype(int), labels=[0, 1]).ravel()
    ref = mm["metrics_recommended"]["confusion_matrix"]
    same = (tn, fp, fn, tp) == (ref["tn"], ref["fp"], ref["fn"], ref["tp"])
    ok &= same
    msgs.append(f"{v}: TN/FP/FN/TP={tn}/{fp}/{fn}/{tp} (ref {ref['tn']}/{ref['fp']}/{ref['fn']}/{ref['tp']}), sens={tp/(tp+fn):.4f}")
record("TC-07", "Reproduksi confusion matrix data uji (split 80:20, seed 42)",
       "154 baris uji, ambang rekomendasi",
       "Confusion matrix sama dengan yang tersimpan di artefak", " | ".join(msgs), ok)

TC-07 [PASS] versi_a: TN/FP/FN/TP=64/36/6/48 (ref 64/36/6/48), sens=0.8889 | versi_b: TN/FP/FN/TP=54/46/12/42 (ref 54/46/12/42), sens=0.7778


### TC-08 — Versi B tidak bergantung pada Glucose & Insulin

In [10]:
pB1, _ = predict("versi_b", pat)
pB2, _ = predict("versi_b", {**pat, "Glucose": 300, "Insulin": 900})
record("TC-08", "Versi B mengabaikan Glucose & Insulin (triase non-lab)",
       "Glukosa/Insulin diubah ekstrem, fitur lain tetap",
       "Probabilitas Versi B tidak berubah", f"p={pB1:.4f} vs p={pB2:.4f}", abs(pB1 - pB2) < 1e-12)

TC-08 [PASS] p=0.6381 vs p=0.6381


### TC-09 — Nilai ekstrem & data kosong

In [11]:
ext   = {"Pregnancies":17, "Glucose":199, "BloodPressure":122, "SkinThickness":99, "Insulin":846, "BMI":67.1, "DiabetesPedigreeFunction":2.42, "Age":81}
empty = {"Pregnancies":0, "Glucose":0, "BloodPressure":0, "SkinThickness":0, "Insulin":0, "BMI":0, "DiabetesPedigreeFunction":0.3, "Age":21}
pe, _ = predict("versi_a", ext)
pz, _ = predict("versi_a", empty)
record("TC-09", "Nilai ekstrem (maks dataset) & semua kolom lab kosong",
       "Maks dataset; semua kolom lab = 0", "Probabilitas valid 0-1, tidak error/NaN",
       f"ekstrem p={pe:.4f}; kosong p={pz:.4f}",
       0 <= pe <= 1 and 0 <= pz <= 1 and not np.isnan(pz))

TC-09 [PASS] ekstrem p=1.0000; kosong p=0.2281


### TC-10 — Input tidak valid (teks, kosong, usia negatif)
Skenario ini **diharapkan menemukan celah**: model tidak memeriksa rentang nilai, sehingga validasi harus dibuat di lapisan web.

In [12]:
errors = []
for label, bad in {"teks": {**pat, "Age": "abc"}, "None": {**pat, "Age": None}}.items():
    try:
        predict("versi_a", bad)
        errors.append(f"{label}: tidak ditolak")
    except ValueError:
        errors.append(f"{label}: ditolak")
neg_p, _ = predict("versi_a", {**pat, "Age": -5})
errors.append(f"Age=-5 diterima p={neg_p:.3f} (perlu validasi rentang di UI)")
record("TC-10", "Input tidak valid (teks, kosong, usia negatif)", "Age='abc'; Age=None; Age=-5",
       "Semua ditolak oleh lapisan validasi", " | ".join(errors),
       all("ditolak" in e for e in errors))

TC-10 [FAIL] teks: ditolak | None: ditolak | Age=-5 diterima p=0.611 (perlu validasi rentang di UI)


## 4. Rekap Log Pengujian

In [13]:
out = pd.DataFrame(log)
out.to_csv("log_pengujian.csv", index=False, encoding="utf-8-sig")

pd.set_option("display.max_colwidth", None)
print("Ringkasan status:", out["Status"].value_counts().to_dict())
out

Ringkasan status: {'PASS': 9, 'FAIL': 1}


,ID,Skenario,Input,Hasil Diharapkan,Hasil Aktual,Status,Waktu
0,TC-01,Integritas artefak model (joblib & JSON),Muat 2 file artefak,Semua 7 pemeriksaan struktur lolos,PERINGATAN versi sklearn: artefak 1.6.1 vs terpasang 1.9.1 (model gagal prediksi tanpa shim). 7/7 lolos,PASS,2026-11-25 23:12:33
1,TC-02,Validasi dataset & penanganan nilai 0,pima-indians-diabetes.csv (768 baris),768 baris; 500 neg/268 pos; nilai 0 -> NaN pada 5 kolom,"768 baris; 500 neg/268 pos; NaN: {'Glucose': 5, 'BloodPressure': 35, 'SkinThickness': 227, 'Insulin': 374, 'BMI': 11}",PASS,2026-11-25 23:12:35
2,TC-03,Konsistensi prediksi joblib vs rumus logistik dari JSON (pasien contoh notebook),"{'Pregnancies': 4, 'Glucose': 148, 'BloodPressure': 76, 'SkinThickness': 30, 'Insulin': 0, 'BMI': 33.6, 'DiabetesPedigreeFunction': 0.627, 'Age': 45}",Selisih probabilitas < 0.0001 untuk Versi A & B,"A: sklearn/manual=(0.7757, 0.7757); B: (0.6381, 0.6381)",PASS,2026-11-25 23:12:38
3,TC-04,Input 0 pada kolom lab diperlakukan sebagai data hilang,Insulin=0 vs Insulin=NaN (fitur lain sama),Probabilitas identik,p(0)=0.7757; p(NaN)=0.7757,PASS,2026-11-25 23:12:40
4,TC-05,Kasus tipikal berisiko tinggi vs rendah (Versi A),Baris 3 CSV (Glukosa 183) vs baris 2 CSV (Glukosa 85),Tinggi -> rujuk (1); Rendah -> tidak rujuk (0),Tinggi p=0.864 -> 1; Rendah p=0.059 -> 0,PASS,2026-11-25 23:12:42
5,TC-06,Konsistensi ambang keputusan & monotonisitas terhadap Glukosa,"Glukosa 100-140 (langkah 1), fitur lain tetap",Label = (p >= 0.35); p naik seiring glukosa,label konsisten=True; monoton naik=True,PASS,2026-11-25 23:12:43
6,TC-07,"Reproduksi confusion matrix data uji (split 80:20, seed 42)","154 baris uji, ambang rekomendasi",Confusion matrix sama dengan yang tersimpan di artefak,"versi_a: TN/FP/FN/TP=64/36/6/48 (ref 64/36/6/48), sens=0.8889 | versi_b: TN/FP/FN/TP=54/46/12/42 (ref 54/46/12/42), sens=0.7778",PASS,2026-11-25 23:12:45
7,TC-08,Versi B mengabaikan Glucose & Insulin (triase non-lab),"Glukosa/Insulin diubah ekstrem, fitur lain tetap",Probabilitas Versi B tidak berubah,p=0.6381 vs p=0.6381,PASS,2026-11-25 23:12:48
8,TC-09,Nilai ekstrem (maks dataset) & semua kolom lab kosong,Maks dataset; semua kolom lab = 0,"Probabilitas valid 0-1, tidak error/NaN",ekstrem p=1.0000; kosong p=0.2281,PASS,2026-11-25 23:12:49
9,TC-10,"Input tidak valid (teks, kosong, usia negatif)",Age='abc'; Age=None; Age=-5,Semua ditolak oleh lapisan validasi,teks: ditolak | None: ditolak | Age=-5 diterima p=0.611 (perlu validasi rentang di UI),FAIL,2026-11-25 23:12:51


## 5. Kesimpulan & Tindak Lanjut
- **TC-01–TC-09 PASS:** artefak konsisten, perhitungan model dapat direproduksi, dan Versi B benar-benar tidak memakai Glucose/Insulin.
- **TC-10 FAIL (temuan):** model menerima nilai di luar rentang (mis. usia negatif). Tambahkan validasi rentang di web (contoh: usia 21–120, glukosa 40–500, BMI 10–70).
- **Versi scikit-learn:** pasang versi yang sama dengan saat pelatihan (lihat sel pertama) agar model dapat dimuat tanpa penyesuaian.
- **Disclaimer:** hasil model adalah alat bantu skrining, bukan diagnosis medis.